%md
# Error Handling Demo — Databricks Notebook Logger

This notebook demonstrates how the logger behaves when a notebook encounters an error. All executed code, outputs, warnings, and the error message are automatically captured in the log file before logging stops. The log file is exported to the Databricks Workspace and both the log file and HTML snapshot to your designated SFTP location.

### Start Logging
Initalize the logger and proceed with notebook code as normal

In [0]:
# Install and import the logging module
%pip install nb-audit-logger
from nb_audit_logger import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.3/818.3 kB 23.3 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Not uninstalling cffi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d0156b3b-7b61-4c86-b8d2-9050b2ae90c7
    Can't uninstall 'cffi'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Credential persistence path chosen: /home/spark-d0156b3b-7b61-4c86-b8d2-90/.sftp_saved_creds_hmcooper.json


In [0]:
start_logging(output_directory="/output/logs")

user                 : hmcooper@berkeley.edu
run start timestamp  : 2026-03-08T13:19:00+00:00
notebook             : /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/error_notebook
log file             : /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/error_notebook.log
cluster id           : 0308-131839-2s65c9b2-v2n
python               : 3.12.3
ipython              : 8.32.0
spark                : 3.5.2
NOTE: Logging started



SFTP host:  eu-central-1.sftpcloud.io

SFTP port:  22

Username:  02c8d61a24e6412286d4996b868e185c

Password (hidden):  [REDACTED]

Validating SFTP credentials...
Connecting to eu-central-1.sftpcloud.io:22 as 02c8d61a24e6412286d4996b868e185c ...
Connection established.
Closing connection.
SFTP authentication successful.


Save your credentials for this cluster only (cleared when it restarts)? [y/N]:  y

Credentials saved to /home/spark-d0156b3b-7b61-4c86-b8d2-90/.sftp_saved_creds_hmcooper.json (mode 0o600).
SFTP: local log path will be /home/spark-d0156b3b-7b61-4c86-b8d2-90/hmcooper/error_notebook.log


In [0]:
# Imports
import pyspark
from pyspark import sql, SparkConf
from pyspark.sql import functions as f
from pyspark.sql.types import *
import os, sys
from pyspark.sql.window import Window
import warnings
from datetime import date

In [0]:
# Create sample patient visit data
visit_data = [
    ("P001", date(2023, 8, 21), "99213", "Hypertension"),
    ("P002", date(2023, 8, 22), "99214", "Diabetes"),
    ("P003", date(2023, 8, 23), "99212", "Annual checkup"),
    ("P001", date(2023, 8, 24), "99215", "Hypertension"),
    ("P004", date(2023, 8, 25), "99213", "Asthma"),
    ("P002", date(2023, 8, 26), "99213", "Diabetes"),
]

patient_visits = spark.createDataFrame(visit_data, ["patient_id", "visit_date", "procedure_code", "diagnosis"])
patient_visits.show()

+----------+----------+--------------+--------------+
|patient_id|visit_date|procedure_code|     diagnosis|
+----------+----------+--------------+--------------+
|      P001|2023-08-21|         99213|  Hypertension|
|      P002|2023-08-22|         99214|      Diabetes|
|      P003|2023-08-23|         99212|Annual checkup|
|      P001|2023-08-24|         99215|  Hypertension|
|      P004|2023-08-25|         99213|        Asthma|
|      P002|2023-08-26|         99213|      Diabetes|
+----------+----------+--------------+--------------+



## Error Handling
Upon encountering an error in the notebook, the logger immediately stops execution, appends a full error summary to the log, and uploads both the log file and HTML snapshot to the Workspace and SFTP location - ensuring a complete audit trail is preserved even when a notebook fails

In [0]:
raise ValueError("This is a test error.")

[LOGGING] ERROR detected in cell: ValueError: This is a test error.

NOTEBOOK DID NOT SUCCESSFULLY RUN DUE TO ERROR
ERROR: ValueError: This is a test error.
run end timestamp    : 2026-03-08T13:19:27+00:00
total runtime        : 26.51 seconds
NOTE: Logging stopped. Uploaded to Workspace → /Repos/hmcooper@berkeley.edu/Databricks-Notebook-Logger/demo/error_notebook.log
SFTP: wrote local log → /home/spark-d0156b3b-7b61-4c86-b8d2-90/hmcooper/error_notebook.log
Opening SFTP session...
Connecting to eu-central-1.sftpcloud.io:22 as 02c8d61a24e6412286d4996b868e185c ...
Connection established.
Ensuring remote directory exists: /output/logs
Uploading file:
 remote: /output/logs/error_notebook.log
Upload completed successfully.
Closing connection.
SFTP: uploaded → /output/logs/error_notebook.log
SFTP: deleted local log → /home/spark-d0156b3b-7b61-4c86-b8d2-90/hmcooper/error_notebook.log
SFTP: wrote local HTML → /home/spark-d0156b3b-7b61-4c86-b8d2-90/hmcooper/error_notebook.html
Opening SFTP sessi

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-5552966729954244>, line 1
----> 1 raise ValueError("This is a test error.")

ValueError: This is a test error.

In [0]:
# Create sample patient demographics table
demographics_data = [
    ("P001", "Male", 45, "Commercial"),
    ("P002", "Female", 62, "Medicare"),
    ("P003", "Male", 34, "Commercial"),
    ("P004", "Female", 28, "Medicaid")
]

demographics = spark.createDataFrame(demographics_data, ["patient_id", "gender", "age", "insurance"])
demographics.show()

In [0]:
# Join sample visit with demographic data
patient_analysis = patient_visits.join(demographics, on="patient_id", how="left")
patient_analysis.show()

In [0]:
# Create sample summary statistics by demographics
summary = patient_analysis.groupBy("gender", "insurance") \
    .agg(
        f.count("visit_date").alias("total_visits"),
        f.countDistinct("patient_id").alias("unique_patients"),
        f.avg("age").alias("avg_age")
    ) \
    .orderBy("gender", "insurance")

summary.show(truncate=False)

In [0]:
# Create temporary view to query with SQL
patient_visits.createOrReplaceTempView("patient_visits")

In [0]:
# Query from spark.sql() will show up in the log
spark.sql("""
create or replace temp view diagnosis_summary as
select diagnosis, count(*) as patient_count
from patient_visits
group by diagnosis
""")

In [0]:
# Use .show(truncate=False) to execute query and display output in the log file
# truncate=False ensures no values are truncated/cut off in the log file
spark.sql("""
select *
from diagnosis_summary
where diagnosis = 'Hypertension'
    or diagnosis = 'Diabetes'
""").show(truncate=False)

In [0]:
# Use log_df to display table contents in the log file
log_df("diagnosis_summary", limit=10)

In [0]:
# User warning
warnings.warn("This is a test warning", UserWarning)

In [0]:
stop_logging()